In [1]:
import sys
import os
sys.path.append('/work/mherre/mothernet/ticl')
from ticl.models.decoders import MLPModelDecoder, GradTreeDecoder

import torch

# Decoder V2

In [2]:
# --- configuration ---
batch_size = 2
n_samples = 30      # number of training examples (token length)
n_features = 10
n_classes = 3
emsize = 32
tree_depth = 3

# --- dummy transformer output (like encoder output) ---
x = torch.randn(n_samples, batch_size, emsize)
y_src = torch.randint(0, n_classes, (n_samples, batch_size))

# --- instantiate decoder ---
decoder = GradTreeDecoder(
    emsize=emsize,
    hidden_size=64,
    n_out=n_classes,
    decoder_type="output_attention",
    embed_dim=128,
    decoder_hidden_layers=2,
    nhead=2,
    in_size=n_features,
    tree_depth=tree_depth,
)

# --- forward pass ---
I_logits, T, L = decoder(x, y_src)

# --- sanity checks ---
n_nodes = 2**tree_depth - 1
n_leaves = 2**tree_depth

assert I_logits.shape == (batch_size, n_nodes, n_features)
assert T.shape == (batch_size, n_nodes, n_features)
assert L.shape == (batch_size, n_leaves, n_classes)

print("✅ GradTreeDecoder forward() successful.")
print(f"I_logits: {I_logits.shape}, T: {T.shape}, L: {L.shape}")
print(f"Mean/std of outputs: {I_logits.mean():.3f}/{I_logits.std():.3f}")

I_logits.sum().backward()
grad_norm = sum(p.grad.abs().sum() for p in decoder.parameters() if p.grad is not None)
print(f"Total grad norm: {grad_norm.item():.2f}")

n_nodes = 2**tree_depth - 1
n_leaves = 2**tree_depth
expected = n_nodes * (2 * n_features) + n_leaves * n_classes
actual = decoder.num_output_layer_weights
assert expected == actual
print("Parameter count correct:", actual)

✅ GradTreeDecoder forward() successful.
I_logits: torch.Size([2, 7, 10]), T: torch.Size([2, 7, 10]), L: torch.Size([2, 8, 3])
Mean/std of outputs: -0.012/0.085
Total grad norm: 954.37
Parameter count correct: 164


In [3]:
import torch
from ticl.models.mothernet import MotherNet

# --- Hyperparameters ---
n_features = 10
n_classes = 3
tree_depth = 3
batch_size = 2       # two datasets in parallel
n_train = 16
n_test = 8

# --- Dummy data ---
x = torch.randn(n_train + n_test, batch_size, n_features)
y = torch.randint(0, n_classes, (n_train + n_test, batch_size))
single_eval_pos = n_train  # split position

# --- Model ---
model = MotherNet(
    n_out=n_classes,
    emsize=64,
    nhead=4,
    nhid_factor=2,
    nlayers=2,
    n_features=n_features,
    child_model="gradtree",
    tree_depth=tree_depth,
    decoder_type="output_attention",  # MotherNet default for tabular
)

# --- Forward pass ---
with torch.no_grad():
    out = model((x, y), single_eval_pos=single_eval_pos)

print("✅ Forward pass completed!")
print("Output shape:", out.shape)
print("Sample output:", out[0, 0])

I_logits, T, L = model.decoder(model.inner_forward(model.encoder(x[:n_train])), y[:n_train])
print("Feature indices per node:", I_logits.argmax(-1)[0])
print("Thresholds:", T.gather(2, I_logits.argmax(-1).unsqueeze(-1)).squeeze(-1)[0])
print("Leaf logits:", L[0])


Number of parameters in backbone:  66944


✅ Forward pass completed!
Output shape: torch.Size([8, 2, 3])
Sample output: tensor([ 0.0341, -0.0100, -0.0902])
Feature indices per node: tensor([8, 6, 0, 7, 6, 4, 6])
Thresholds: tensor([-0.0113,  0.0531,  0.0390, -0.0310, -0.0136, -0.0254,  0.0186],
       grad_fn=<SelectBackward0>)
Leaf logits: tensor([[ 0.0341, -0.0100, -0.0902],
        [-0.0741, -0.0673,  0.0189],
        [ 0.0005,  0.0172,  0.0125],
        [-0.0241,  0.0142, -0.0512],
        [-0.0454, -0.0231, -0.0618],
        [ 0.0152, -0.0704,  0.0508],
        [ 0.0383,  0.0750,  0.0624],
        [ 0.0156,  0.0531,  0.0823]], grad_fn=<SelectBackward0>)


## MotherNet Testing

In [4]:
import torch

depth = 2
n_features = 3
n_out = 2

# Create simple thresholds that separate x[..., 0]
model = MotherNet(
    n_out=n_out, emsize=32, nhead=2, nhid_factor=2, nlayers=1,
    n_features=n_features, child_model="gradtree", tree_depth=depth
)

x = torch.zeros(6, 1, n_features)
x[:3, 0, 0] = -1   # left
x[3:, 0, 0] = +1   # right
y = torch.randint(0, n_out, (6, 1))

with torch.no_grad():
    out = model((x, y), single_eval_pos=3)

print("Output logits:\n", out.squeeze())

I_logits, T, L = model.decoder(model.inner_forward(model.encoder(x[:3])), y[:3])

print("Feature logits (argmax):", I_logits.argmax(-1))
print("Thresholds (first node):", T[:, 0, :])
print("Leaf logits:", L)

out = model((x, y), single_eval_pos=3)
loss = out.mean()
loss.backward()

# ✅ Should show nonzero grads for:
print("Grad for decoder.mlp:", model.decoder.mlp[0].weight.grad.abs().mean())
print("Grad for encoder:", model.encoder.weight.grad.abs().mean())

Number of parameters in backbone:  8544


Output logits:
 tensor([[0.0707, 0.0203],
        [0.0707, 0.0203],
        [0.0707, 0.0203]])
Feature logits (argmax): tensor([[0, 1, 0]])
Thresholds (first node): tensor([[ 0.1193, -0.0038,  0.0672]], grad_fn=<SliceBackward0>)
Leaf logits: tensor([[[ 0.0809, -0.0803],
         [ 0.0165,  0.0424],
         [-0.0375, -0.0898],
         [ 0.0707,  0.0203]]], grad_fn=<ViewBackward0>)
Grad for decoder.mlp: tensor(0.0014)
Grad for encoder: tensor(0.0012)


In [5]:
import torch

torch.manual_seed(42)

# === Configuration ===
n_features = 5
n_out = 2
tree_depth = 3
n_train = 4
n_test = 3
batch_size = 1  # number of datasets (meta-batch)

# === Create synthetic dataset ===
# Training + test split
x_train = torch.randn(n_train, batch_size, n_features)
y_train = torch.randint(0, n_out, (n_train, batch_size))
x_test = torch.randn(n_test, batch_size, n_features)
y_test = torch.randint(0, n_out, (n_test, batch_size))

# Concatenate for MotherNet input (same convention as in your forward)
x_all = torch.cat([x_train, x_test], dim=0)
y_all = torch.cat([y_train, y_test], dim=0)

single_eval_pos = n_train  # number of training samples per dataset

# === Initialize MotherNet with GradTree child ===
model = MotherNet(
    n_out=n_out,
    emsize=32,
    nhead=2,
    nhid_factor=2,
    nlayers=1,
    n_features=n_features,
    child_model="gradtree",
    tree_depth=tree_depth,
    decoder_type="output_attention",
    tabpfn_zero_weights=False,
)

# === Forward pass ===
out = model((x_all, y_all), single_eval_pos=single_eval_pos)
print("✅ Forward pass output shape:", out.shape)
print("Output logits:\n", out.squeeze())

# === Check for NaNs ===
assert not torch.isnan(out).any(), "❌ Output contains NaNs!"

# === Compute dummy loss and backward ===
loss = out.mean()
loss.backward()

# === Check gradient flow ===
encoder_grad = model.encoder.weight.grad
decoder_grad = model.decoder.mlp[0].weight.grad

print("✅ Encoder grad mean:", encoder_grad.abs().mean().item())
print("✅ Decoder grad mean:", decoder_grad.abs().mean().item())

assert encoder_grad.abs().sum() > 0, "❌ Encoder has no gradients!"
assert decoder_grad.abs().sum() > 0, "❌ Decoder has no gradients!"

print("\n🎉 All tests passed — GradTree is fully differentiable!")


Number of parameters in backbone:  8544
✅ Forward pass output shape: torch.Size([3, 1, 2])
Output logits:
 tensor([[-0.0548,  0.0383],
        [ 0.0947,  0.0582],
        [ 0.1013,  0.0299]], grad_fn=<SqueezeBackward0>)
✅ Encoder grad mean: 0.0011616727570071816
✅ Decoder grad mean: 0.0007276774267666042

🎉 All tests passed — GradTree is fully differentiable!


In [6]:
from ticl.models.mothernet import entmax15

I_logits, T, L = model.decoder(model.inner_forward(model.encoder(x_all)), y_all[:single_eval_pos])
pi = entmax15(I_logits, dim=-1)
print("Feature selection sparsity:", (pi == 0).float().mean().item())

x = torch.zeros(1, 5)  # all logits = 0
p = entmax15(x, dim=-1)
print(p)
print("Sum =", p.sum())


Feature selection sparsity: 0.0
tensor([[0.2000, 0.2000, 0.2000, 0.2000, 0.2000]])
Sum = tensor(1.)


In [8]:
from ticl.prediction.mothernet import extract_gradtree_model
config = {}
params = extract_gradtree_model(model, config, x_train, y_train, device="cpu")

for k, v in params.items():
    print(k, v.shape)

RuntimeError: Tensors must have same number of dimensions: got 3 and 2

# Decoder V1

In [11]:

# Example parameters (adjust as needed)
emsize = 512
n_out = 10
hidden_size = 1024
decoder_type = 'output_attention'
predicted_hidden_layer_size = 256
embed_dim = 2048
decoder_hidden_layers = 1
nhead = 4
predicted_hidden_layers = 1
weight_embedding_rank = None
low_rank_weights = False
in_size = 100

decoder = MLPModelDecoder(
    emsize=emsize,
    n_out=n_out,
    hidden_size=hidden_size,
    decoder_type=decoder_type,
    predicted_hidden_layer_size=predicted_hidden_layer_size,
    embed_dim=embed_dim,
    decoder_hidden_layers=decoder_hidden_layers,
    nhead=nhead,
    predicted_hidden_layers=predicted_hidden_layers,
    weight_embedding_rank=weight_embedding_rank,
    low_rank_weights=low_rank_weights,
    in_size=in_size
)

In [12]:
# Simulate transformer embeddings instead
batch_size = 2
n_samples = 50
x = torch.randn(n_samples, batch_size, emsize)  # fake embeddings
y_src = torch.randint(0, n_out, (n_samples, batch_size))  # fake labels

# Now use the normal forward method
result = decoder(x, y_src)  # This will work

In [13]:
# Examine what result contains
print("Type of result:", type(result))
print("Length of result:", len(result))
print("\nStructure of result:")
for i, item in enumerate(result):
    if isinstance(item, tuple):
        print(f"result[{i}]: tuple with {len(item)} elements")
        for j, sub_item in enumerate(item):
            if hasattr(sub_item, 'shape'):
                print(f"  [{j}]: tensor with shape {sub_item.shape}")
            else:
                print(f"  [{j}]: {type(sub_item)}")
    else:
        if hasattr(item, 'shape'):
            print(f"result[{i}]: tensor with shape {item.shape}")
        else:
            print(f"result[{i}]: {type(item)}")

# If it's the expected MLP weights structure, let's examine them
if len(result) >= 1 and isinstance(result[0], tuple):
    print("\nFirst layer (should be bias1, weights1):")
    b1, w1 = result[0]
    print(f"b1 shape: {b1.shape}")  # should be [batch, hidden_size]
    print(f"w1 shape: {w1.shape}")  # should be [batch, in_size, hidden_size]
    
    if len(result) >= 2:
        print("\nSecond layer (should be bias2, weights2):")
        b2, w2 = result[1]
        print(f"b2 shape: {b2.shape}")  # should be [batch, n_out]
        print(f"w2 shape: {w2.shape}")  # should be [batch, hidden_size, n_out]

Type of result: <class 'list'>
Length of result: 2

Structure of result:
result[0]: tuple with 2 elements
  [0]: tensor with shape torch.Size([2, 256])
  [1]: tensor with shape torch.Size([2, 100, 256])
result[1]: tuple with 2 elements
  [0]: tensor with shape torch.Size([2, 10])
  [1]: tensor with shape torch.Size([2, 256, 10])

First layer (should be bias1, weights1):
b1 shape: torch.Size([2, 256])
w1 shape: torch.Size([2, 100, 256])

Second layer (should be bias2, weights2):
b2 shape: torch.Size([2, 10])
w2 shape: torch.Size([2, 256, 10])


In [14]:
# Test the first generated MLP
test_input = torch.randn(5, 100)  # 5 test samples, 100 features
h = torch.matmul(test_input, w1[0]) + b1[0]  # Use first MLP's weights
h = torch.relu(h)
output = torch.matmul(h, w2[0]) + b2[0]
print("First MLP output:", output.shape)  # Should be [5, 10]

First MLP output: torch.Size([5, 10])


# GradTree

In [15]:
# Test GradTreeDecoder

# Create GradTreeDecoder instance
grad_tree_decoder = GradTreeDecoder(
    emsize=emsize,
    n_out=n_out,
    hidden_size=hidden_size,
    decoder_type=decoder_type,
    embed_dim=embed_dim,
    decoder_hidden_layers=decoder_hidden_layers,
    decoder_activation='relu',
    in_size=in_size,
    tree_depth=3  # Creates 2^3-1=7 internal nodes, 2^3=8 leaves
)

print("GradTreeDecoder parameters:")
print(f"Tree depth: {grad_tree_decoder.tree_depth}")
print(f"Internal nodes: {grad_tree_decoder.n_nodes}")
print(f"Leaves: {grad_tree_decoder.n_leaves}")
print(f"Total output parameters: {grad_tree_decoder.num_output_layer_weights}")

# Test with same simulated data
grad_result = grad_tree_decoder(x, y_src)

print("\nGradTree result structure:")
print(f"Type: {type(grad_result)}")
print(f"Length: {len(grad_result)}")

for i, tensor in enumerate(grad_result):
    if hasattr(tensor, 'shape'):
        print(f"grad_result[{i}]: tensor with shape {tensor.shape}")
    else:
        print(f"grad_result[{i}]: {type(tensor)}")

# The result should be (I, T, L) tuples
if len(grad_result) == 3:
    I, T, L = grad_result
    print(f"\nFeature indices (I): {I.shape}")  # [batch, n_nodes, in_size]
    print(f"Thresholds (T): {T.shape}")         # [batch, n_nodes, in_size] 
    print(f"Leaf outputs (L): {L.shape}")       # [batch, n_leaves, n_out]
    
    print(f"\nFirst batch - Feature indices sample:")
    print(f"Node 0 feature selection: {I[0, 0, :5]}")  # First 5 features for first node
    print(f"Node 0 thresholds: {T[0, 0, :5]}")         # Thresholds for first 5 features
    print(f"Leaf 0 class distribution: {L[0, 0]}")     # Class probabilities for first leaf

GradTreeDecoder parameters:
Tree depth: 3
Internal nodes: 7
Leaves: 8
Total output parameters: 1480

GradTree result structure:
Type: <class 'tuple'>
Length: 3
grad_result[0]: tensor with shape torch.Size([2, 7, 100])
grad_result[1]: tensor with shape torch.Size([2, 7, 100])
grad_result[2]: tensor with shape torch.Size([2, 8, 10])

Feature indices (I): torch.Size([2, 7, 100])
Thresholds (T): torch.Size([2, 7, 100])
Leaf outputs (L): torch.Size([2, 8, 10])

First batch - Feature indices sample:
Node 0 feature selection: tensor([-0.0128,  0.0336, -0.0067,  0.0172, -0.0071], grad_fn=<SliceBackward0>)
Node 0 thresholds: tensor([ 0.0307, -0.0232, -0.0122, -0.0325, -0.0025], grad_fn=<SliceBackward0>)
Leaf 0 class distribution: tensor([-0.0087,  0.0094, -0.0073, -0.0374, -0.0461, -0.0509,  0.0274,  0.0209,
        -0.0241,  0.0501], grad_fn=<SelectBackward0>)


In [19]:
import torch
import torch.nn.functional as F

# === (1) Define straight-through helper functions ===
def straight_through_round(x):
    hard = torch.round(x)
    return hard + x - x.detach()

def straight_through_one_hot(logits):
    probs = F.softmax(logits, dim=-1)
    hard = torch.zeros_like(probs)
    hard.scatter_(2, probs.argmax(-1, keepdim=True), 1.0)
    return hard + probs - probs.detach()

# === (2) GradTree forward pass ===
def gradtree_forward(x_test, I, T, L):
    batch, n_nodes, n_features = I.shape
    n_leaves, n_out = L.shape[1], L.shape[2]
    tree_depth = int(torch.log2(torch.tensor(n_leaves)).item())

    feature_onehot = straight_through_one_hot(I)
    thresholds = (T * feature_onehot).sum(-1)

    x_feat = x_test.unsqueeze(0).unsqueeze(2).expand(batch, -1, n_nodes, -1)
    fsel = feature_onehot.unsqueeze(1).expand(-1, x_test.size(0), -1, -1)
    x_sel = (x_feat * fsel).sum(-1)
    t_sel = thresholds.unsqueeze(1).expand(-1, x_test.size(0), -1)

    s = torch.sigmoid(x_sel - t_sel)
    s_hard = straight_through_round(s)

    device = x_test.device
    leaf_bits = ((torch.arange(n_leaves, device=device).unsqueeze(1)
                 >> torch.arange(tree_depth, device=device)) & 1).float()

    node_ids = torch.arange(n_nodes, device=device)
    node_ids_per_depth = []
    idx = 0
    for d in range(tree_depth):
        n_at_depth = 2 ** d
        node_ids_per_depth.append(node_ids[idx:idx + n_at_depth])
        idx += n_at_depth

    n_test = x_test.size(0)
    path_prob = torch.ones(batch, n_test, n_leaves, device=device)

    for depth, nodes_at_depth in enumerate(node_ids_per_depth):
        s_depth = s_hard[:, :, nodes_at_depth]
        l_bit = leaf_bits[:, depth]
        for k, node_id in enumerate(nodes_at_depth):
            match = torch.where(
                l_bit[None, None, :] == 0,
                s_depth[:, :, k:k+1],
                1 - s_depth[:, :, k:k+1],
            )
            path_prob *= match

    h = torch.einsum("btl,blo->bto", path_prob, L)
    return h


In [21]:
I = torch.randn(2, 7, 100, requires_grad=True)
T = torch.randn(2, 7, 100, requires_grad=True)
L = torch.randn(2, 8, 10, requires_grad=True)
x_test = torch.randn(5, 100)


In [22]:
# Test data: 5 samples, 100 features (matches in_size)
x_test = torch.randn(5, 100)

# Forward pass through GradTree
h = gradtree_forward(x_test, I, T, L)
print("Output logits shape:", h.shape)  # expect [batch=2, n_test=5, n_out=10]

# Backward pass to test differentiability
loss = h.mean()
loss.backward()

print("\nGradient check:")
print("I.grad mean:", I.grad.abs().mean().item())
print("T.grad mean:", T.grad.abs().mean().item())
print("L.grad mean:", L.grad.abs().mean().item())


Output logits shape: torch.Size([2, 5, 10])

Gradient check:
I.grad mean: 4.5457982196239755e-05
T.grad mean: 3.7459085433511063e-05
L.grad mean: 0.0006249999860301614
